# broadcasting-rules — ex10: end-to-end scaled-dot-product attention via 4-axis broadcasting

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. Running the final beacon cell reports progress against the `Numpy: Vectorization and broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcasting-rules`** (exercise 10). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting rules — quick refresher

Two shapes broadcast right-aligned: from the trailing axis backward, each pair of dims must be equal, OR one of them must be 1, OR one of them must be missing. Multi-axis broadcasting (e.g. `(B, 1, T, T)` with `(B, H, T, T)`) is what lets a single padding mask apply to every attention head without copying.

### Exercise 10 — end-to-end scaled-dot-product attention via 4-axis broadcasting

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Bloom level: Create
> LO: Combine batch-axis, head-axis, and padding-mask broadcasting in a single end-to-end scaled-dot-product attention forward pass, materialising no intermediates beyond what broadcasting supplies.
> Keywords: attention, multi-head, padding-mask, scaled-dot-product, integrative, ml-adjacent
> ```

**KCs targeted:** `predict-broadcast-shape`, `axis-insertion-via-unsqueeze`

Existing ex1–ex9 covered the broadcasting rule, row/column broadcast, axis insertion, outer product, pairwise distance, attention SCORES (ex7), per-channel bias, and a silent-broadcast trap. This drill is the *end-to-end attention* integration: compute the FULL scaled-dot-product attention output (not just the scores), and broadcast a per-sample padding mask across all heads.

Implement `ex10_attention(Q, K, V, pad_mask)`:

1. `Q`, `K`, `V` all have shape `(B, H, T, D)` — batch, heads, sequence, head-dim.
2. `pad_mask` has shape `(B, T)` — `True` where the token is a real token, `False` where it is padding.
3. Compute attention scores: `S = Q @ K.transpose(-1, -2) / sqrt(D)` (shape `(B, H, T, T)`).
4. Broadcast the padding mask **from `(B, T)` to `(B, 1, 1, T)`** so it applies across all heads and all query positions; set padded KEY positions to `-inf`.
5. Softmax over the last axis to get attention weights `(B, H, T, T)`.
6. Weighted sum: `out = weights @ V` (shape `(B, H, T, D)`).
7. Return `(out, weights)` — both `float32`.

Key broadcasting moves:
- `(B, T)` → `(B, 1, 1, T)` via two `unsqueeze`s (head + query-position) — broadcast across `H` and the query axis.
- Padded-row safety: softmax over an all-`-inf` row produces NaN; the test tolerates that for fully-padded rows but rejects NaN in real rows.

In [ ]:
def ex10_attention(Q: Tensor, K: Tensor, V: Tensor, pad_mask: Tensor):
    import math
    B, H, T, D = Q.shape
    scores = Q @ K.transpose(-1, -2) / math.sqrt(D)         # (B, H, T, T)
    # (B, T) -> (B, 1, 1, T) so it broadcasts over heads + query axis.
    key_mask = pad_mask.unsqueeze(1).unsqueeze(1)            # (B, 1, 1, T)
    scores = scores.masked_fill(~key_mask, float('-inf'))
    weights = t.softmax(scores, dim=-1)                      # (B, H, T, T)
    out = weights @ V                                        # (B, H, T, D)
    return out.to(t.float32), weights.to(t.float32)


<details><summary>Solution</summary>

```python
def ex10_attention(Q: Tensor, K: Tensor, V: Tensor, pad_mask: Tensor):
    import math
    B, H, T, D = Q.shape
    scores = Q @ K.transpose(-1, -2) / math.sqrt(D)         # (B, H, T, T)
    # (B, T) -> (B, 1, 1, T) so it broadcasts over heads + query axis.
    key_mask = pad_mask.unsqueeze(1).unsqueeze(1)            # (B, 1, 1, T)
    scores = scores.masked_fill(~key_mask, float('-inf'))
    weights = t.softmax(scores, dim=-1)                      # (B, H, T, T)
    out = weights @ V                                        # (B, H, T, D)
    return out.to(t.float32), weights.to(t.float32)
```

**Two `unsqueeze`s map `(B, T)` onto `(B, H, T, T)`.** Position 1 (head axis) and position 2 (query axis) are inserted as size-1 dims; right-align broadcasting then expands both to their full size. One `pad_mask` of `B * T` bools controls `B * H * T * T` attention entries — broadcasting buys you a `H * T` reduction.

**Why we mask the KEY axis, not the query axis.** A padded query position is a row of the attention matrix that we'll throw away downstream (its output is irrelevant). A padded KEY position must NEVER contribute to ANY query's output, so we zero it on the column axis. Masking columns to `-inf` before softmax → softmax sends them to 0.

**Contrast with ex7.** ex7 computed batched scores with shape-trace debug. This drill adds the V multiply, the mask, and the softmax — the whole attention block — and verifies against `torch.nn.functional.scaled_dot_product_attention`. That's the broadcasting payoff: 6 lines of torch is the entire attention layer.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex10',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()